### 00 LIBRERIAS

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from glob import glob
# Aplicar configuraciones de visualización total de Pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

from pathlib import Path
import glob

import warnings
warnings.filterwarnings("ignore")
import plotly.express as px

import func_negocio as func

### 01 EXTRACT

In [3]:
bd_locales = r"input\Productividad 2026 v4 --- ep.xlsx"
df_locales = pd.read_excel(bd_locales, sheet_name="Locales")

In [4]:
bd_2025 = r"input\bd_ventas\Export_VentaProdBI_2025.xlsx"
df_2025 = pd.read_excel(bd_2025)


df_2025["codigo_local"] = df_2025["Codigo Local"]
df_2025["tipo_venta"] = df_2025["Tipo Venta"]
df_2025["nombre_area"] = df_2025["nombre_area"]
df_2025["nombre_division"] = df_2025["nombre_division"]
df_2025["VtaNeta"] = df_2025["VtaNeta"]
df_2025["VtaUnidad"] = df_2025["VtaUnidad"]
df_2025["VtaNetaPpto"] = df_2025["VtaNetaPpto"]

df_2025 = df_2025[~df_2025["nombre_division"].isnull()]

In [5]:
lista_archivos = [
    "Export_VentaProdBI_202601.xlsx",
    "Export_VentaProdBI_202602.xlsx",
    "Export_VentaProdBI_202603.xlsx",
    "Export_EstimarMesActual_20260430.xlsx",
    "Export_EstimarMesActual_202605.xlsx",
    "Export_EstimarMesActual_20260611.xlsx",
]
ruta = r"input\bd_ventas"

rutas_completas = [os.path.join(ruta, archivo) for archivo in lista_archivos]
dfs = [pd.read_excel(p) for p in rutas_completas]
df_2026 = pd.concat(dfs, ignore_index=True)

df_2026["codigo_local"] = df_2026["Codigo Local"]
df_2026["tipo_venta"] = df_2026["Tipo Venta"]
df_2026["nombre_area"] = df_2026["nombre_area"]
df_2026["nombre_division"] = df_2026["nombre_division"]
df_2026["VtaNeta"] = df_2026["VtaNeta"]
df_2026["VtaUnidad"] = df_2026["VtaUnidad"]
df_2026["VtaNetaPpto"] = df_2026["VtaNetaPpto"]

df_2026 = df_2026[~df_2026["nombre_division"].isnull()]
df_2026["Anio"] = df_2026["Anio"].fillna(df_2026["Fecha"].dt.year)
df_2026["NroMes"] = df_2026["NroMes"].fillna(df_2026["Fecha"].dt.month)


In [6]:
df_ventas_all = pd.concat([df_2025, df_2026])


In [7]:
bd_ppto_2026 = r"input\bd_ventas\Export_PptoVenta.xlsx"
df_ppto_2026 = pd.read_excel(bd_ppto_2026)

df_ppto_2026["codigo_local"] = df_ppto_2026["codigo_local"]
df_ppto_2026["tipo_venta"] = df_ppto_2026["DpPv_Lv4"]
df_ppto_2026["nombre_area"] = df_ppto_2026["Area"]
df_ppto_2026["nombre_division"] = df_ppto_2026["Division"]
df_ppto_2026["VtaNeta"] = df_ppto_2026["Ppto"]
df_ppto_2026["VtaUnidad"] = df_ppto_2026["Ppto"]
df_ppto_2026["VtaNetaPpto"] = df_ppto_2026["Ppto"]
df_ppto_2026["Anio"] = df_ppto_2026["Periodo"].astype(str).str[0:4]
df_ppto_2026["NroMes"] = df_ppto_2026["Periodo"].astype(str).str[-2:]

df_ppto_2026 = df_ppto_2026[~df_ppto_2026["nombre_division"].isnull()]
df_ppto_2026["Anio"] = df_ppto_2026["Anio"].astype(int)
df_ppto_2026["NroMes"] = df_ppto_2026["NroMes"].astype(int)

### 02 TRANSFORM

In [8]:
df_ventas_all["codigo_local"] = df_ventas_all["codigo_local"].astype(str)
df_locales["codigo_local"] = df_locales["codigo_local"].astype(str)

df_2025x = df_ventas_all.merge(df_locales[["codigo_local", "_Tienda"]], on="codigo_local", how="inner")
df_2025x["area"] = np.where(df_2025x["nombre_area"] == "ELECTRO", "ELECTRO",
                           np.where(df_2025x["nombre_division"]== "FRESCOS", "FRESCOS",
                                    np.where(df_2025x["nombre_division"]=="NON FOOD", "NONFOOD", "ABARROTES")))

df_ventas_final = func.logica_area_puesto( df_2025x, func.config_reglas_prod)

In [9]:
df_ppto_2026["codigo_local"] = df_ppto_2026["codigo_local"].astype(str)
df_locales["codigo_local"] = df_locales["codigo_local"].astype(str)

df_ppto_2026x = df_ppto_2026.merge(df_locales[["codigo_local", "_Tienda"]], on="codigo_local", how="inner")
df_ppto_2026x["area"] = np.where(df_ppto_2026x["nombre_area"] == 'ELECTRO A11', "ELECTRO",
                           np.where(df_ppto_2026x["nombre_division"]== 'FRESCOS D2', "FRESCOS",
                                    np.where(df_ppto_2026x["nombre_division"]=='NON FOOD D3', "NONFOOD", "ABARROTES")))

df_ppto_2026x["tipo_venta"] = np.where(df_ppto_2026x["tipo_venta"]=='E-Commerce', "Venta Ecommerce",
                                      np.where(df_ppto_2026x["tipo_venta"]=='Tienda', "Venta POS",
                                               np.where(df_ppto_2026x["tipo_venta"]=='Last Mile', "Last Mile",df_ppto_2026x["tipo_venta"] )))

df_ppto_2026xx = func.logica_area_puesto( df_ppto_2026x, func.config_reglas_prod)
df_ppto_2026xx["Ppto2026"] = df_ppto_2026xx["Vta."]
df_ppto_2026z = df_ppto_2026xx[['_Tienda', 'Anio', 'NroMes', 'area','Ppto2026']]

### 03 LOAD

In [10]:
df_ventas_final["_Periodo"] = df_ventas_final["Anio"].astype(int).astype(str)+df_ventas_final["NroMes"].astype(int).astype(str).str.zfill(2)
df_ventas_final = df_ventas_final.rename(columns={"area": "_Tipo"})

In [11]:
df_ppto_2026z["_Periodo"] = df_ppto_2026z["Anio"].astype(int).astype(str)+df_ppto_2026z["NroMes"].astype(int).astype(str).str.zfill(2)
df_ppto_2026z = df_ppto_2026z.rename(columns={"area": "_Tipo"})

In [12]:
df_ppto_2026z["_Tipo"].unique()

array(['ELECTRO', 'FRESCOS', 'ALMACEN FRESCOS', 'ABARROTES', 'CAJAS',
       'RECEPCION', 'ALMACEN', 'INVENTARIOS', 'MULTIFUNCIONAL'],
      dtype=object)

In [13]:
df_ventas_final.to_parquet(r"output\VentasReal.parquet")
df_ppto_2026z.to_parquet(r"output\VentasPPTO.parquet")